In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/duongquanganh/processed-data-v6/processed_val.csv
/kaggle/input/datasets/duongquanganh/processed-data-v6/processed_train.csv
/kaggle/input/datasets/duongquanganh/processed-data-v6/processed_test.csv


In [2]:
import csv

In [3]:
TRAIN_PATH = "/kaggle/input/datasets/duongquanganh/processed-data-v6/processed_train.csv"
VAL_PATH = "/kaggle/input/datasets/duongquanganh/processed-data-v6/processed_val.csv"
TEST_PATH = "/kaggle/input/datasets/duongquanganh/processed-data-v6/processed_test.csv"

In [4]:
def process(path):
    raw_reviews = []
    cleaned_reviews = []
    ws_reviews = []
    full_labels = []
    aspect_labels = []
    aspect_category_labels = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_review = row.get("raw_reviews").strip()
            raw_reviews.append(raw_review)

            cleaned_review = row.get("cleaned_reviews").strip()
            cleaned_reviews.append(cleaned_review)

            ws_review = row.get("ws_reviews").strip()
            ws_reviews.append(ws_review)
            
            labels = row.get("full_labels").strip()
            full_labels.append(labels)

            review_aspects = row.get("aspect_labels").strip()
            review_aspect_categories = row.get("aspect_category_labels").strip()
            aspect_labels.append(review_aspects)
            aspect_category_labels.append(review_aspect_categories)

    return raw_reviews, cleaned_reviews, ws_reviews, full_labels, aspect_labels, aspect_category_labels

In [5]:
train_raw_reviews, train_cleaned_reviews, train_ws_reviews, train_labels, train_aspect_labels, train_aspect_category_labels = process(TRAIN_PATH)
val_raw_reviews, val_cleaned_reviews, val_ws_reviews, val_labels, val_aspect_labels, val_aspect_category_labels = process(VAL_PATH)
test_raw_reviews, test_cleaned_reviews, test_ws_reviews, test_labels, test_aspect_labels, test_aspect_category_labels = process(TEST_PATH)

len(train_raw_reviews), len(val_raw_reviews), len(test_raw_reviews)

(2961, 1290, 500)

In [6]:
!pip install -q transformers tqdm

In [7]:
import json
import math
import random
import re
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 9e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 140
PATIENCE = 140
WARMUP_RATIO = 0.0
THRESHOLD_ENTITY = 0.5
THRESHOLD_ASPECT = 0.5
LOSS_WEIGHTS = {"entity": 2.0, "aspect": 3.0, "polarity": 1.0}

OUTPUT_DIR = Path("/kaggle/working/hierarchical_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

Device: cuda


In [8]:
REQUIRED_COLUMNS = ["raw_reviews", "cleaned_reviews", "ws_reviews", "full_labels", "aspect_labels", "aspect_category_labels"]

def validate_schema(path: str, required_columns: List[str]):
    df = pd.read_csv(path, encoding="utf-8-sig")
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    return df

train_df_raw = validate_schema(TRAIN_PATH, REQUIRED_COLUMNS)
val_df_raw = validate_schema(VAL_PATH, REQUIRED_COLUMNS)
test_df_raw = validate_schema(TEST_PATH, REQUIRED_COLUMNS)

print("Schema OK")
print("Train/Val/Test:", len(train_df_raw), len(val_df_raw), len(test_df_raw))

def split_pipe_labels(value: str) -> List[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    return [x.strip() for x in text.split("|") if x.strip()]

FULL_LABEL_PATTERN = re.compile(r"\{\s*([^,{}]+)\s*,\s*([^{}]+?)\s*\}")

def parse_full_label(full_label: str) -> List[Tuple[str, str]]:
    if full_label is None or (isinstance(full_label, float) and math.isnan(full_label)):
        return []
    text = str(full_label).strip()
    if not text:
        return []
    pairs = []
    for aspect_category, sentiment in FULL_LABEL_PATTERN.findall(text):
        pairs.append((aspect_category.strip(), sentiment.strip().lower()))
    return pairs

def normalize_sentiment(label: str) -> str:
    s = (label or "").strip().lower()
    if s in {"positive", "negative", "neutral"}:
        return s
    return "neutral"

def debug_raw_label_quality(df: pd.DataFrame, split_name: str):
    empty_aspect_category = 0
    empty_aspect = 0
    empty_full = 0
    parse_failed = 0
    mismatch_non_empty = 0

    for _, row in df.iterrows():
        aspect_category_text = row.get("aspect_category_labels", "")
        aspect_text = row.get("aspect_labels", "")
        full_text = row.get("full_labels", "")

        ac_list = split_pipe_labels(aspect_category_text)
        a_list = split_pipe_labels(aspect_text)
        full_pairs = parse_full_label(full_text)
        full_cats = [ac for ac, _ in full_pairs]

        if not ac_list:
            empty_aspect_category += 1
        if not a_list:
            empty_aspect += 1
        if str(full_text).strip() == "":
            empty_full += 1
        if str(full_text).strip() != "" and len(full_pairs) == 0:
            parse_failed += 1
        if ac_list and full_cats:
            # Aspect categories present in full_label but missing from aspect_category_label column.
            miss = [x for x in full_cats if x not in ac_list]
            if miss:
                mismatch_non_empty += 1

    n = len(df)
    print(f"[{split_name}] empty aspect_category_label: {empty_aspect_category}/{n}")
    print(f"[{split_name}] empty aspect_label: {empty_aspect}/{n}")
    print(f"[{split_name}] empty full_label: {empty_full}/{n}")
    print(f"[{split_name}] full_label parse_failed (non-empty but no pair): {parse_failed}/{n}")
    print(f"[{split_name}] rows with mismatch between aspect_category_label and full_label: {mismatch_non_empty}/{n}")

debug_raw_label_quality(train_df_raw, "train")
debug_raw_label_quality(val_df_raw, "val")
debug_raw_label_quality(test_df_raw, "test")

Schema OK
Train/Val/Test: 2961 1290 500
[train] empty aspect_category_label: 0/2961
[train] empty aspect_label: 0/2961
[train] empty full_label: 0/2961
[train] full_label parse_failed (non-empty but no pair): 0/2961
[train] rows with mismatch between aspect_category_label and full_label: 0/2961
[val] empty aspect_category_label: 0/1290
[val] empty aspect_label: 0/1290
[val] empty full_label: 0/1290
[val] full_label parse_failed (non-empty but no pair): 0/1290
[val] rows with mismatch between aspect_category_label and full_label: 0/1290
[test] empty aspect_category_label: 0/500
[test] empty aspect_label: 0/500
[test] empty full_label: 0/500
[test] full_label parse_failed (non-empty but no pair): 0/500
[test] rows with mismatch between aspect_category_label and full_label: 0/500


In [9]:
SENTIMENT_SPACE = ["negative", "neutral", "positive"]
sentiment2id = {s: i for i, s in enumerate(SENTIMENT_SPACE)}
id2sentiment = {i: s for s, i in sentiment2id.items()}

def build_records(df: pd.DataFrame) -> List[Dict]:
    records = []
    for _, row in df.iterrows():
        ws_review = str(row.get("ws_reviews", "") or "").strip()
        full_label = str(row.get("full_labels", "") or "").strip()
        aspect_text = row.get("aspect_labels", "")
        aspect_category_text = row.get("aspect_category_labels", "")

        full_pairs = parse_full_label(full_label)
        aspect_categories = split_pipe_labels(aspect_category_text)
        if not aspect_categories:
            aspect_categories = [ac for ac, _ in full_pairs]

        aspects = split_pipe_labels(aspect_text)
        if not aspects and aspect_categories:
            aspects = [ac.split("#", 1)[0] for ac in aspect_categories]

        polarity_by_aspect_category = OrderedDict()
        for ac, pol in full_pairs:
            polarity_by_aspect_category[ac] = normalize_sentiment(pol)

        records.append({
            "ws_review": ws_review,
            "full_label": full_label,
            "aspects": list(OrderedDict.fromkeys(aspects)),
            "aspect_categories": list(OrderedDict.fromkeys(aspect_categories)),
            "polarity_by_aspect_category": polarity_by_aspect_category,
        })
    return records

train_records = build_records(train_df_raw)
val_records = build_records(val_df_raw)
test_records = build_records(test_df_raw)

all_records = train_records + val_records + test_records
all_aspects = sorted({a for r in all_records for a in r["aspects"] if a})
all_aspect_categories = sorted({ac for r in all_records for ac in r["aspect_categories"] if ac})

aspect2id = {a: i for i, a in enumerate(all_aspects)}
id2aspect = {i: a for a, i in aspect2id.items()}
aspect_category2id = {ac: i for i, ac in enumerate(all_aspect_categories)}
id2aspect_category = {i: ac for ac, i in aspect_category2id.items()}

taxonomy = {
    "aspects": all_aspects,
    "aspect_categories": all_aspect_categories,
    "sentiments": SENTIMENT_SPACE,
    "aspect2id": aspect2id,
    "aspect_category2id": aspect_category2id,
    "sentiment2id": sentiment2id,
}

with open(OUTPUT_DIR / "taxonomy.json", "w", encoding="utf-8") as f:
    json.dump(taxonomy, f, ensure_ascii=False, indent=2)

print("Taxonomy frozen")
print("num_aspects:", len(all_aspects))
print("num_aspect_categories:", len(all_aspect_categories))

def debug_mapping_consistency(records: List[Dict], split_name: str):
    unknown_aspects = 0
    unknown_aspect_categories = 0
    for rec in records:
        for a in rec["aspects"]:
            if a not in aspect2id:
                unknown_aspects += 1
        for ac in rec["aspect_categories"]:
            if ac not in aspect_category2id:
                unknown_aspect_categories += 1
    print(f"[{split_name}] unknown aspects in mapping: {unknown_aspects}")
    print(f"[{split_name}] unknown aspect categories in mapping: {unknown_aspect_categories}")

def debug_label_distribution(records: List[Dict], split_name: str):
    n = len(records)
    cat_count = {ac: 0 for ac in all_aspect_categories}
    ent_count = {a: 0 for a in all_aspects}

    for rec in records:
        for ac in rec["aspect_categories"]:
            if ac in cat_count:
                cat_count[ac] += 1
        for a in rec["aspects"]:
            if a in ent_count:
                ent_count[a] += 1

    cat_rate = {k: (v / (n + 1e-9)) for k, v in cat_count.items()}
    ent_rate = {k: (v / (n + 1e-9)) for k, v in ent_count.items()}

    top_cat = sorted(cat_rate.items(), key=lambda x: x[1], reverse=True)[:10]
    top_ent = sorted(ent_rate.items(), key=lambda x: x[1], reverse=True)[:10]
    zero_cat = [k for k, v in cat_count.items() if v == 0]
    zero_ent = [k for k, v in ent_count.items() if v == 0]

    print(f"[{split_name}] top-10 aspect_category positive rates:")
    for k, v in top_cat:
        print(f"  {k}: rate={v:.4f}, support={cat_count[k]}")

    print(f"[{split_name}] top-10 entity positive rates:")
    for k, v in top_ent:
        print(f"  {k}: rate={v:.4f}, support={ent_count[k]}")

    print(f"[{split_name}] aspect_category with zero support: {len(zero_cat)}")
    print(f"[{split_name}] entity with zero support: {len(zero_ent)}")

debug_mapping_consistency(train_records, "train")
debug_mapping_consistency(val_records, "val")
debug_mapping_consistency(test_records, "test")

debug_label_distribution(train_records, "train")
debug_label_distribution(val_records, "val")
debug_label_distribution(test_records, "test")

Taxonomy frozen
num_aspects: 6
num_aspect_categories: 12
[train] unknown aspects in mapping: 0
[train] unknown aspect categories in mapping: 0
[val] unknown aspects in mapping: 0
[val] unknown aspect categories in mapping: 0
[test] unknown aspects in mapping: 0
[test] unknown aspect categories in mapping: 0
[train] top-10 aspect_category positive rates:
  FOOD#QUALITY: rate=0.8977, support=2658
  FOOD#STYLE&OPTIONS: rate=0.5897, support=1746
  FOOD#PRICES: rate=0.4732, support=1401
  RESTAURANT#GENERAL: rate=0.2888, support=855
  SERVICE#GENERAL: rate=0.2685, support=795
  AMBIENCE#GENERAL: rate=0.2489, support=737
  LOCATION#GENERAL: rate=0.1260, support=373
  RESTAURANT#PRICES: rate=0.0888, support=263
  RESTAURANT#MISCELLANEOUS: rate=0.0510, support=151
  DRINKS#QUALITY: rate=0.0385, support=114
[train] top-10 entity positive rates:
  FOOD: rate=0.9672, support=2864
  RESTAURANT: rate=0.3705, support=1097
  SERVICE: rate=0.2685, support=795
  AMBIENCE: rate=0.2489, support=737
  LOC

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ABSAHierDataset(Dataset):
    def __init__(self, records: List[Dict], tokenizer, max_length: int):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(
            rec["ws_review"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        y_entity = torch.zeros(len(aspect2id), dtype=torch.float)
        for a in rec["aspects"]:
            if a in aspect2id:
                y_entity[aspect2id[a]] = 1.0

        y_aspect = torch.zeros(len(aspect_category2id), dtype=torch.float)
        for ac in rec["aspect_categories"]:
            if ac in aspect_category2id:
                y_aspect[aspect_category2id[ac]] = 1.0

        y_polarity = torch.full((len(aspect_category2id),), -100, dtype=torch.long)
        for ac, pol in rec["polarity_by_aspect_category"].items():
            if ac in aspect_category2id:
                y_polarity[aspect_category2id[ac]] = sentiment2id[normalize_sentiment(pol)]

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "y_entity": y_entity,
            "y_aspect": y_aspect,
            "y_polarity": y_polarity,
            "text": rec["ws_review"],
        }

train_dataset = ABSAHierDataset(train_records, tokenizer, MAX_LENGTH)
val_dataset = ABSAHierDataset(val_records, tokenizer, MAX_LENGTH)
test_dataset = ABSAHierDataset(test_records, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)

print(len(train_dataset), len(val_dataset), len(test_dataset))

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2961 1290 500


In [11]:
map_entity_2_aspect = [1, 3, 3, 1, 3, 1]
# ambience: 1, drinks: 3, foods: 3, location: 1, restaurant: 3, service: 1

class HierarchicalABSA(nn.Module):

    def __init__(self, model_name: str, n_entity: int, n_aspect: int, n_sentiment: int):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.n_entity = n_entity
        self.n_aspect = n_aspect
        self.n_sentiment = n_sentiment
        self.map_entity_2_aspect = map_entity_2_aspect
        
        self.entity_head = nn.Linear(hidden, n_entity)
        self.entity_bridge = nn.Sequential(
            nn.Linear(n_entity, hidden * 2),
            nn.GELU(),
            nn.Linear(hidden * 2, hidden),
        )

        self.register_buffer('entity_masks', torch.eye(n_entity))

        self.aspect_heads = nn.ModuleList([
            nn.Linear(hidden * 2, n_aspects) for n_aspects in map_entity_2_aspect
        ])

        self.aspect_bridges = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_aspects, hidden * 2),
                nn.GELU(),
                nn.Linear(hidden * 2, hidden * 2)
            ) for n_aspects in map_entity_2_aspect
        ])

        self.polarity_heads = nn.ModuleList([
            nn.Linear(hidden * 3, n_aspects * n_sentiment) for n_aspects in map_entity_2_aspect
        ])

        self.drop = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        x = self.drop(out.last_hidden_state[:, 0, :])
        
        entity_logits = self.entity_head(x)
        entity_probs = torch.sigmoid(entity_logits)

        aspect_logits_list = []
        polarity_logits_list = []
        aspect_probs_list = []
        
        for i in range(self.n_entity):
            # 1. Hard Mask Entity
            mask_i = self.entity_masks[i]
            masked_probs = entity_probs * mask_i
            
            # 2. Trích xuất ngữ cảnh Entity cho nhánh i
            entity_ctx_i = self.entity_bridge(masked_probs) + x
            h2_i = torch.cat([x, entity_ctx_i], dim=-1)

            # 3. Dự đoán Aspect (Kích thước output phụ thuộc map_entity_2_aspect[i])
            aspect_logits_i = self.aspect_heads[i](h2_i)
            aspect_probs_i = torch.sigmoid(aspect_logits_i)
            
            aspect_logits_list.append(aspect_logits_i)
            aspect_probs_list.append(aspect_probs_i)

            # 4. Trích xuất ngữ cảnh Aspect cho nhánh i
            aspect_ctx_i = self.aspect_bridges[i](aspect_probs_i) + h2_i
            h3_i = torch.cat([x, aspect_ctx_i], dim=-1)

            # 5. Dự đoán Polarity cho nhánh i
            n_aspects_i = self.map_entity_2_aspect[i]
            # Reshape thành [batch_size, n_aspects_i, n_sentiment]
            polarity_logits_i = self.polarity_heads[i](h3_i).view(-1, n_aspects_i, self.n_sentiment)
            polarity_logits_list.append(polarity_logits_i)

        # Gộp (Concatenate) dọc theo chiều aspect (dim=1) để đồng nhất với định dạng loss function
        # aspect_logits_concat: [batch_size, 12]
        aspect_logits_concat = torch.cat(aspect_logits_list, dim=1)
        aspect_probs_concat = torch.cat(aspect_probs_list, dim=1)
        
        # polarity_logits_concat: [batch_size, 12, n_sentiment]
        polarity_logits_concat = torch.cat(polarity_logits_list, dim=1)

        return {
            "entity_logits": entity_logits,
            "aspect_logits": aspect_logits_concat,
            "polarity_logits": polarity_logits_concat,
            "entity_probs": entity_probs,
            "aspect_probs": aspect_probs_concat,
        }

ce_loss = nn.CrossEntropyLoss(ignore_index=-100)


def compute_global_loss(outputs, y_entity, y_aspect, y_polarity):
    l_entity = F.binary_cross_entropy_with_logits(outputs["entity_logits"], y_entity)
    l_aspect = F.binary_cross_entropy_with_logits(outputs["aspect_logits"], y_aspect)

    pol_logits = outputs["polarity_logits"].reshape(-1, len(SENTIMENT_SPACE))

    pol_target = y_polarity.reshape(-1)

    l_polarity = ce_loss(pol_logits, pol_target)

    total = (
        LOSS_WEIGHTS["entity"] * l_entity
        + LOSS_WEIGHTS["aspect"] * l_aspect
        + LOSS_WEIGHTS["polarity"] * l_polarity
    )

    return total, {
        "entity": l_entity.item(),
        "aspect": l_aspect.item(),
        "polarity": l_polarity.item(),
        "global": total.item(),
    }


def micro_f1_from_binary(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-9):

    tp = ((y_true == 1) & (y_pred == 1)).sum().item()

    fp = ((y_true == 0) & (y_pred == 1)).sum().item()

    fn = ((y_true == 1) & (y_pred == 0)).sum().item()

    precision = tp / (tp + fp + eps)

    recall = tp / (tp + fn + eps)

    f1 = 2 * precision * recall / (precision + recall + eps)

    return precision, recall, f1


def aspect_category_java_style_metrics(

    y_aspect: torch.Tensor,

    y_polarity: torch.Tensor,

    aspect_probs: torch.Tensor,

    polarity_logits: torch.Tensor,

    threshold_aspect: float = 0.5,

    eps: float = 1e-9,

) -> Dict[str, Dict[str, float]]:

    """Mirror SAEvaluate.java counting for Aspect#Category and Aspect#Category#Polarity."""

    aspect_true = y_aspect.long()

    aspect_pred = (aspect_probs >= threshold_aspect).long()

    pol_true = y_polarity.long()

    pol_pred = polarity_logits.argmax(dim=-1).long()

    total_gold = int((aspect_true == 1).sum().item())

    total_pred = int((aspect_pred == 1).sum().item())

    correct_aspect = int(((aspect_true == 1) & (aspect_pred == 1)).sum().item())

    correct_aspect_polarity = int(

        ((aspect_true == 1) & (aspect_pred == 1) & (pol_true != -100) & (pol_pred == pol_true)).sum().item()

    )

    p_aspect = correct_aspect / (total_pred + eps)

    r_aspect = correct_aspect / (total_gold + eps)

    f1_aspect = 2 * p_aspect * r_aspect / (p_aspect + r_aspect + eps)

    p_ap = correct_aspect_polarity / (total_pred + eps)

    r_ap = correct_aspect_polarity / (total_gold + eps)

    f1_ap = 2 * p_ap * r_ap / (p_ap + r_ap + eps)

    return {

        "aspect_category": {

            "precision": p_aspect,

            "recall": r_aspect,

            "micro_f1": f1_aspect,

            "correct": correct_aspect,

            "predicted": total_pred,

            "gold": total_gold,

        },

        "aspect_category_polarity": {

            "precision": p_ap,

            "recall": r_ap,

            "micro_f1": f1_ap,

            "correct": correct_aspect_polarity,

            "predicted": total_pred,

            "gold": total_gold,

        },

    }

In [12]:
def run_epoch(model, loader, optimizer=None, scheduler=None, train_mode=True):
    if train_mode:
        model.train()
    else:
        model.eval()

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available() and train_mode)

    losses = []
    ent_true_all, ent_pred_all = [], []
    asp_true_all, asp_pred_all, asp_probs_all = [], [], []
    pol_logits_all, pol_true_all = [], []

    pbar = tqdm(loader, desc="train" if train_mode else "eval", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        y_entity = batch["y_entity"].to(device)
        y_aspect = batch["y_aspect"].to(device)
        y_polarity = batch["y_polarity"].to(device)

        with torch.set_grad_enabled(train_mode):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available() and train_mode):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss, loss_items = compute_global_loss(outputs, y_entity, y_aspect, y_polarity)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()

        losses.append(loss_items)

        ent_pred = (outputs["entity_probs"] >= THRESHOLD_ENTITY).long().detach().cpu()
        asp_pred = (outputs["aspect_probs"] >= THRESHOLD_ASPECT).long().detach().cpu()
        asp_probs = outputs["aspect_probs"].detach().cpu()
        ent_true = y_entity.long().detach().cpu()
        asp_true = y_aspect.long().detach().cpu()

        ent_true_all.append(ent_true)
        ent_pred_all.append(ent_pred)
        asp_true_all.append(asp_true)
        asp_pred_all.append(asp_pred)
        asp_probs_all.append(asp_probs)
        pol_logits_all.append(outputs["polarity_logits"].detach().cpu())
        pol_true_all.append(y_polarity.detach().cpu())

    avg_loss = {k: float(np.mean([x[k] for x in losses])) for k in losses[0].keys()}

    ent_true_cat = torch.cat(ent_true_all, dim=0)
    ent_pred_cat = torch.cat(ent_pred_all, dim=0)
    asp_true_cat = torch.cat(asp_true_all, dim=0)
    asp_pred_cat = torch.cat(asp_pred_all, dim=0)
    asp_probs_cat = torch.cat(asp_probs_all, dim=0)
    pol_logits_cat = torch.cat(pol_logits_all, dim=0)
    pol_true_cat = torch.cat(pol_true_all, dim=0)

    ent_p, ent_r, ent_f1 = micro_f1_from_binary(ent_true_cat, ent_pred_cat)
    asp_p, asp_r, asp_f1 = micro_f1_from_binary(asp_true_cat, asp_pred_cat)
    java_style = aspect_category_java_style_metrics(
        y_aspect=asp_true_cat,
        y_polarity=pol_true_cat,
        aspect_probs=asp_probs_cat,
        polarity_logits=pol_logits_cat,
        threshold_aspect=THRESHOLD_ASPECT,
    )

    metrics = {
        "loss": avg_loss,
        "entity": {"precision": ent_p, "recall": ent_r, "micro_f1": ent_f1},
        "aspect": {"precision": asp_p, "recall": asp_r, "micro_f1": asp_f1},
        "aspect_category": java_style["aspect_category"],
        "aspect_category_polarity": java_style["aspect_category_polarity"],
    }
    return metrics

In [13]:
model = HierarchicalABSA(
    model_name=MODEL_NAME,
    n_entity=len(aspect2id),
    n_aspect=len(aspect_category2id),
    n_sentiment=len(SENTIMENT_SPACE),
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,)

history = []
best_test_aspect_f1 = -1.0
best_epoch = -1
bad_epochs = 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer, scheduler, train_mode=True)
    # val_metrics = run_epoch(model, val_loader, train_mode=False)
    test_metrics = run_epoch(model, test_loader, train_mode=False)

    row = {
        "epoch": epoch,
        "train": train_metrics,
        # "val": val_metrics,
        "test": test_metrics,
    }
    history.append(row)

    # val_aspect_f1 = val_metrics["aspect"]["micro_f1"]
    # val_entity_f1 = val_metrics["entity"]["micro_f1"]
    # val_ac_polarity_f1 = val_metrics["aspect_category_polarity"]["micro_f1"]
    test_aspect_p = test_metrics["aspect"]["precision"]
    test_aspect_r = test_metrics["aspect"]["recall"]
    test_aspect_f1 = test_metrics["aspect"]["micro_f1"]

    test_entity_p = test_metrics["entity"]["precision"]
    test_entity_r = test_metrics["entity"]["recall"]
    test_entity_f1 = test_metrics["entity"]["micro_f1"]

    test_ac_polarity_p = test_metrics["aspect_category_polarity"]["precision"]
    test_ac_polarity_r = test_metrics["aspect_category_polarity"]["recall"]
    test_ac_polarity_f1 = test_metrics["aspect_category_polarity"]["micro_f1"]
    
    print(
        f"Epoch {epoch:02d} | train_global_loss={train_metrics['loss']['global']:.4f}\n"
        f"  [Entity]                   P: {test_entity_p:.4f} | R: {test_entity_r:.4f} | F1: {test_entity_f1:.4f}\n"
        f"  [Aspect#Category]          P: {test_aspect_p:.4f} | R: {test_aspect_r:.4f} | F1: {test_aspect_f1:.4f}\n"
        f"  [Aspect#Category#Polarity] P: {test_ac_polarity_p:.4f} | R: {test_ac_polarity_r:.4f} | F1: {test_ac_polarity_f1:.4f}\n"
        f"--------------------------------------------------------------------------------"
    )

    if test_aspect_f1 > best_test_aspect_f1:
        best_test_aspect_f1 = test_aspect_f1
        best_epoch = epoch
        bad_epochs = 0

        ckpt_dir = OUTPUT_DIR / "best_checkpoint"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), ckpt_dir / "model.pt")
        tokenizer.save_pretrained(ckpt_dir)

        with open(ckpt_dir / "training_config.json", "w", encoding="utf-8") as f:
            json.dump({
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "threshold_entity": THRESHOLD_ENTITY,
                "threshold_aspect": THRESHOLD_ASPECT,
                "loss_weights": LOSS_WEIGHTS,
                "best_epoch": best_epoch,
                "best_test_aspect_micro_f1": best_test_aspect_f1,
            }, f, ensure_ascii=False, indent=2)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

with open(OUTPUT_DIR / "history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print("Best epoch:", best_epoch)
print("Best test Aspect Micro-F1:", round(best_test_aspect_f1, 6))

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

/tmp/ipykernel_58/1399836253.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available() and train_mode)


...

eval:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 139 | train_global_loss=0.0036
  [Entity]                   P: 0.8767 | R: 0.9147 | F1: 0.8953
  [Aspect#Category]          P: 0.8301 | R: 0.8607 | F1: 0.8451
  [Aspect#Category#Polarity] P: 0.7185 | R: 0.7449 | F1: 0.7315
--------------------------------------------------------------------------------


train:   0%|          | 0/186 [00:00<?, ?it/s]

eval:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 140 | train_global_loss=0.0037
  [Entity]                   P: 0.8762 | R: 0.9147 | F1: 0.8950
  [Aspect#Category]          P: 0.8300 | R: 0.8619 | F1: 0.8457
  [Aspect#Category#Polarity] P: 0.7189 | R: 0.7466 | F1: 0.7325
--------------------------------------------------------------------------------
Best epoch: 106
Best test Aspect Micro-F1: 0.848571


In [14]:
best_model = HierarchicalABSA(
    model_name=MODEL_NAME,
    n_entity=len(aspect2id),
    n_aspect=len(aspect_category2id),
    n_sentiment=len(SENTIMENT_SPACE),
).to(device)

best_model.load_state_dict(torch.load(OUTPUT_DIR / "best_checkpoint" / "model.pt", map_location=device))
best_model.eval()

test_metrics = run_epoch(best_model, test_loader, train_mode=False)
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

with open(OUTPUT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_58/1399836253.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available() and train_mode)


eval:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_58/1399836253.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available() and train_mode):


{
  "loss": {
    "entity": 0.7466622088104486,
    "aspect": 0.8826372921466827,
    "polarity": 0.865964287891984,
    "global": 5.007200688123703
  },
  "entity": {
    "precision": 0.874764002516756,
    "recall": 0.9120734908130498,
    "micro_f1": 0.8930292317514914
  },
  "aspect": {
    "precision": 0.8325377883847126,
    "recall": 0.8652335675895555,
    "micro_f1": 0.8485708488815562
  },
  "aspect_category": {
    "precision": 0.8325377883847126,
    "recall": 0.8652335675895555,
    "micro_f1": 0.8485708488815562,
    "correct": 2093,
    "predicted": 2514,
    "gold": 2419
  },
  "aspect_category_polarity": {
    "precision": 0.7211614956242159,
    "recall": 0.74948325754413,
    "micro_f1": 0.7350496650178279,
    "correct": 1813,
    "predicted": 2514,
    "gold": 2419
  }
}


In [15]:
def run_epoch_for_debug_current_error(model, loader, tokenizer):
    model.eval()

    ent_true_all, ent_pred_all, ent_probs_all = [], [], []
    asp_true_all, asp_pred_all, asp_probs_all = [], [], []
    pol_logits_all, pol_true_all = [], []
    input_ids_all = []

    pbar = tqdm(loader, desc="test", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        y_entity = batch["y_entity"].to(device)
        y_aspect = batch["y_aspect"].to(device)
        y_polarity = batch["y_polarity"].to(device)

        with torch.set_grad_enabled(False):
            with torch.cuda.amp.autocast(False):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)


        ent_probs = outputs["entity_probs"].detach().cpu()
        ent_pred = (ent_probs >= THRESHOLD_ENTITY).long()
        
        asp_probs = outputs["aspect_probs"].detach().cpu()
        asp_pred = (asp_probs >= THRESHOLD_ASPECT).long()
        
        ent_true = y_entity.long().detach().cpu()
        asp_true = y_aspect.long().detach().cpu()

        ent_true_all.append(ent_true)
        ent_pred_all.append(ent_pred)
        ent_probs_all.append(ent_probs)
        
        asp_true_all.append(asp_true)
        asp_pred_all.append(asp_pred)
        asp_probs_all.append(asp_probs)
        
        pol_logits_all.append(outputs["polarity_logits"].detach().cpu())
        pol_true_all.append(y_polarity.detach().cpu())

        input_ids_all.append(input_ids.detach().cpu())


    ent_true_cat = torch.cat(ent_true_all, dim=0)
    ent_pred_cat = torch.cat(ent_pred_all, dim=0)
    ent_probs_cat = torch.cat(ent_probs_all, dim=0)
    
    asp_true_cat = torch.cat(asp_true_all, dim=0)
    asp_pred_cat = torch.cat(asp_pred_all, dim=0)
    asp_probs_cat = torch.cat(asp_probs_all, dim=0)
    
    pol_logits_cat = torch.cat(pol_logits_all, dim=0)
    pol_true_cat = torch.cat(pol_true_all, dim=0)

    input_ids_cat = torch.cat(input_ids_all, dim=0)

    ent_p, ent_r, ent_f1 = micro_f1_from_binary(ent_true_cat, ent_pred_cat)
    asp_p, asp_r, asp_f1 = micro_f1_from_binary(asp_true_cat, asp_pred_cat)
    java_style = aspect_category_java_style_metrics(
        y_aspect=asp_true_cat,
        y_polarity=pol_true_cat,
        aspect_probs=asp_probs_cat,
        polarity_logits=pol_logits_cat,
        threshold_aspect=THRESHOLD_ASPECT,
    )

    pol_pred_cat = torch.argmax(pol_logits_cat, dim=-1)

    print("\n" + "="*50)
    print("BẮT ĐẦU IN CÁC TRƯỜNG HỢP PREDICT SAI LỆCH")
    print("="*50)
    
    for i in range(len(asp_true_cat)):
        # Sai Aspect (False Positive hoặc False Negative)
        aspect_mismatch = (asp_pred_cat[i] != asp_true_cat[i])
        
        # Sai Polarity (Dự đoán đúng Aspect nhưng sai cực tính)
        polarity_mismatch = (asp_pred_cat[i] == 1) & (asp_true_cat[i] == 1) & (pol_pred_cat[i] != pol_true_cat[i])

        if aspect_mismatch.any() or polarity_mismatch.any():
            print(aspect2id)
            print(aspect_category2id)
            print(sentiment2id)
            print(f"\n[Sample Index: {i}]")
            decoded_text = tokenizer.decode(input_ids_cat[i], skip_special_tokens=True)
            
            print(f"\n[Sample Index: {i}]")
            print(f"Text Review: {decoded_text}")
            print(f"1. Entity Probs/Logits: \n{ent_probs_cat[i].numpy()}")
            print(f"2. Aspect Category:")
            print(f"   - Probs  : {asp_probs_cat[i].numpy()}")
            print(f"   - Predict: {asp_pred_cat[i].numpy()}")
            print(f"   - Truth  : {asp_true_cat[i].numpy()}")
            print(f"3. Aspect Category Polarity:")
            print(f"   - Logits : \n{pol_logits_cat[i].numpy()}")
            print(f"   - Predict: {pol_pred_cat[i].numpy()}")
            print(f"   - Truth  : {pol_true_cat[i].numpy()}")
    print("\n" + "="*50 + "\n")
    # -----------------------------------------------
    metrics = {
        "entity": {"precision": ent_p, "recall": ent_r, "micro_f1": ent_f1},
        "aspect": {"precision": asp_p, "recall": asp_r, "micro_f1": asp_f1},
        "aspect_category": java_style["aspect_category"],
        "aspect_category_polarity": java_style["aspect_category_polarity"],
    }
    return metrics

In [16]:
best_model = HierarchicalABSA(
    model_name=MODEL_NAME,
    n_entity=len(aspect2id),
    n_aspect=len(aspect_category2id),
    n_sentiment=len(SENTIMENT_SPACE),
).to(device)

best_model.load_state_dict(torch.load(OUTPUT_DIR / "best_checkpoint" / "model.pt", map_location=device))
best_model.eval()

test_metrics = run_epoch_for_debug_current_error(best_model, test_loader, tokenizer)
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

with open(OUTPUT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


test:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_58/460096179.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(False):



BẮT ĐẦU IN CÁC TRƯỜNG HỢP PREDICT SAI LỆCH
{'AMBIENCE': 0, 'DRINKS': 1, 'FOOD': 2, 'LOCATION': 3, 'RESTAURANT': 4, 'SERVICE': 5}
{'AMBIENCE#GENERAL': 0, 'DRINKS#PRICES': 1, 'DRINKS#QUALITY': 2, 'DRINKS#STYLE&OPTIONS': 3, 'FOOD#PRICES': 4, 'FOOD#QUALITY': 5, 'FOOD#STYLE&OPTIONS': 6, 'LOCATION#GENERAL': 7, 'RESTAURANT#GENERAL': 8, 'RESTAURANT#MISCELLANEOUS': 9, 'RESTAURANT#PRICES': 10, 'SERVICE#GENERAL': 11}
{'negative': 0, 'neutral': 1, 'positive': 2}

[Sample Index: 0]

[Sample Index: 0]
Text Review: Đây là 1 trong những quán mà mình thích vì vị trà đậm và thơm cũng như mùi_vị đặc_trưng hơn hẳn những quán khác nè Trà sữa trân_châu sợi - 46k Trà sữa pha khá ngon , vị trà chát và mùi hương khá rõ , không quá ngọt , rất đúng với gu mình Trà đào - 45k Vị trà đào ở đây cũng đặc_biệt hơn hẳn những quán khác , không phải chua_ngọt như_thường thấy mà có mùi trà rất ngon Cà_phê đá xay - 65k Món đá xay ở đây uống cũng ngon không kém trà_nè , mùi_vị thơm hương cà_phê , vị đắng kết_hợp hoàn_hảo v

In [17]:
def predict_texts(model, tokenizer, texts: List[str], threshold_entity=0.5, threshold_aspect=0.5):
    model.eval()
    outputs = []
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(
                text,
                truncation=True,
                max_length=MAX_LENGTH,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask)

            entity_probs = out["entity_probs"][0].cpu()
            aspect_probs = out["aspect_probs"][0].cpu()
            polarity_pred = out["polarity_logits"][0].argmax(dim=-1).cpu()

            pred_entities = [
                id2aspect[i]
                for i, p in enumerate(entity_probs.tolist())
                if p >= threshold_entity
            ]
            pred_aspect_categories = [
                id2aspect_category[i]
                for i, p in enumerate(aspect_probs.tolist())
                if p >= threshold_aspect
            ]
            pred_triplets = []
            for ac in pred_aspect_categories:
                ac_idx = aspect_category2id[ac]
                pol = id2sentiment[int(polarity_pred[ac_idx].item())]
                pred_triplets.append((ac, pol))

            outputs.append({
                "text": text,
                "pred_entities": pred_entities,
                "pred_aspect_categories": pred_aspect_categories,
                "pred_aspect_category_sentiments": pred_triplets,
            })
    return outputs

sample_texts = [
    "Đồ ăn ngon nhưng nhân_viên phục_vụ chậm",
    "Không_gian đẹp, giá hợp_lý và menu đa_dạng.",
    "Vị_trí khó tìm, món ăn bình_thường.",
]

sample_predictions = predict_texts(best_model, tokenizer, sample_texts)
sample_predictions

[{'text': 'Đồ ăn ngon nhưng nhân_viên phục_vụ chậm',
  'pred_entities': ['FOOD', 'SERVICE'],
  'pred_aspect_categories': ['FOOD#QUALITY', 'SERVICE#GENERAL'],
  'pred_aspect_category_sentiments': [('FOOD#QUALITY', 'positive'),
   ('SERVICE#GENERAL', 'negative')]},
 {'text': 'Không_gian đẹp, giá hợp_lý và menu đa_dạng.',
  'pred_entities': ['AMBIENCE', 'FOOD', 'RESTAURANT'],
  'pred_aspect_categories': ['AMBIENCE#GENERAL',
   'FOOD#STYLE&OPTIONS',
   'RESTAURANT#PRICES'],
  'pred_aspect_category_sentiments': [('AMBIENCE#GENERAL', 'positive'),
   ('FOOD#STYLE&OPTIONS', 'positive'),
   ('RESTAURANT#PRICES', 'positive')]},
 {'text': 'Vị_trí khó tìm, món ăn bình_thường.',
  'pred_entities': ['FOOD', 'LOCATION'],
  'pred_aspect_categories': ['FOOD#QUALITY', 'LOCATION#GENERAL'],
  'pred_aspect_category_sentiments': [('FOOD#QUALITY', 'positive'),
   ('LOCATION#GENERAL', 'negative')]}]

In [18]:
def to_english_label(pairs: List[Tuple[str, str]]) -> str:
    if not pairs:
        return ""
    uniq = list(OrderedDict.fromkeys((ac, normalize_sentiment(pol)) for ac, pol in pairs))
    return ", ".join([f"{{{ac}, {pol}}}" for ac, pol in uniq])

def save_split_predictions(records: List[Dict], split_name: str) -> pd.DataFrame:
    texts = [r["ws_review"] for r in records]
    preds = predict_texts(
        best_model,
        tokenizer,
        texts,
        threshold_entity=THRESHOLD_ENTITY,
        threshold_aspect=THRESHOLD_ASPECT,
    )

    rows = []
    for rec, pred in zip(records, preds):
        gold_pairs = parse_full_label(rec.get("full_label", ""))
        pred_pairs = pred.get("pred_aspect_category_sentiments", [])

        gold_label_en = to_english_label(gold_pairs)
        pred_label_en = to_english_label(pred_pairs)

        rows.append({
            "cleaned_review": rec.get("cleaned_review", ""),
            "gold_label_en": gold_label_en,
            "pred_label_en": pred_label_en,
            "gold_raw_full_label": rec.get("full_label", ""),
            "exact_match": int(gold_label_en == pred_label_en),
            "gold_num_pairs": len(gold_pairs),
            "pred_num_pairs": len(pred_pairs),
            "pred_entities": " | ".join(pred.get("pred_entities", [])),
            "pred_aspect_categories": " | ".join(pred.get("pred_aspect_categories", [])),
        })

    df_out = pd.DataFrame(rows)
    out_path = OUTPUT_DIR / f"{split_name}_predictions_english_labels.csv"
    df_out.to_csv(out_path, index=False, encoding="utf-8-sig")

    exact_match_rate = (df_out["exact_match"].mean() * 100.0) if len(df_out) else 0.0
    print(f"[{split_name}] saved: {out_path}")
    print(f"[{split_name}] exact-match: {exact_match_rate:.2f}% ({int(df_out['exact_match'].sum())}/{len(df_out)})")
    return df_out

val_pred_df = save_split_predictions(val_records, "val")
test_pred_df = save_split_predictions(test_records, "test")

display(val_pred_df.head(10))
display(test_pred_df.head(10))

[val] saved: /kaggle/working/hierarchical_outputs/val_predictions_english_labels.csv
[val] exact-match: 15.35% (198/1290)
[test] saved: /kaggle/working/hierarchical_outputs/test_predictions_english_labels.csv
[test] exact-match: 0.80% (4/500)


,cleaned_review,gold_label_en,pred_label_en,gold_raw_full_label,exact_match,gold_num_pairs,pred_num_pairs,pred_entities,pred_aspect_categories
0,,"{FOOD#PRICES, positive}, {FOOD#QUALITY, positive}","{FOOD#PRICES, positive}, {FOOD#QUALITY, positi...","{FOOD#PRICES, positive}, {FOOD#QUALITY, positive}",0,2,3,FOOD,FOOD#PRICES | FOOD#QUALITY | FOOD#STYLE&OPTIONS
1,,"{FOOD#STYLE&OPTIONS, positive}, {RESTAURANT#GE...","{AMBIENCE#GENERAL, positive}, {FOOD#PRICES, po...","{FOOD#STYLE&OPTIONS, positive}, {RESTAURANT#GE...",0,3,7,AMBIENCE | FOOD | LOCATION | RESTAURANT,AMBIENCE#GENERAL | FOOD#PRICES | FOOD#QUALITY ...
2,,"{AMBIENCE#GENERAL, positive}, {FOOD#STYLE&OPTI...","{AMBIENCE#GENERAL, positive}, {FOOD#PRICES, po...","{AMBIENCE#GENERAL, positive}, {FOOD#STYLE&OPTI...",0,5,5,AMBIENCE | FOOD | SERVICE,AMBIENCE#GENERAL | FOOD#PRICES | FOOD#QUALITY ...
3,,"{FOOD#QUALITY, positive}, {RESTAURANT#GENERAL,...","{FOOD#QUALITY, positive}, {RESTAURANT#GENERAL,...","{FOOD#QUALITY, positive}, {RESTAURANT#GENERAL,...",1,2,2,FOOD | RESTAURANT,FOOD#QUALITY | RESTAURANT#GENERAL
4,,"{FOOD#QUALITY, positive}","{FOOD#QUALITY, positive}","{FOOD#QUALITY, positive}",1,1,1,FOOD,FOOD#QUALITY
5,,"{AMBIENCE#GENERAL, neutral}, {FOOD#STYLE&OPTIO...","{AMBIENCE#GENERAL, neutral}, {DRINKS#PRICES, p...","{AMBIENCE#GENERAL, neutral}, {FOOD#STYLE&OPTIO...",0,5,6,AMBIENCE | DRINKS | FOOD | SERVICE,AMBIENCE#GENERAL | DRINKS#PRICES | FOOD#PRICES...
6,,"{LOCATION#GENERAL, positive}, {RESTAURANT#MISC...","{AMBIENCE#GENERAL, positive}, {DRINKS#PRICES, ...","{LOCATION#GENERAL, positive}, {RESTAURANT#MISC...",0,7,7,AMBIENCE | FOOD | LOCATION | RESTAURANT,AMBIENCE#GENERAL | DRINKS#PRICES | FOOD#PRICES...
7,,"{FOOD#QUALITY, positive}, {FOOD#PRICES, positi...","{AMBIENCE#GENERAL, positive}, {FOOD#PRICES, po...","{FOOD#QUALITY, positive}, {FOOD#PRICES, positi...",0,5,5,AMBIENCE | FOOD | RESTAURANT | SERVICE,AMBIENCE#GENERAL | FOOD#PRICES | FOOD#QUALITY ...
8,,"{FOOD#QUALITY, positive}, {FOOD#PRICES, positive}","{FOOD#PRICES, positive}, {FOOD#QUALITY, positive}","{FOOD#QUALITY, positive}, {FOOD#PRICES, positive}",0,2,2,FOOD,FOOD#PRICES | FOOD#QUALITY
9,,"{LOCATION#GENERAL, neutral}, {RESTAURANT#GENER...","{DRINKS#PRICES, positive}, {FOOD#PRICES, posit...","{LOCATION#GENERAL, neutral}, {RESTAURANT#GENER...",0,5,6,DRINKS | FOOD | LOCATION | SERVICE,DRINKS#PRICES | FOOD#PRICES | FOOD#QUALITY | F...


,cleaned_review,gold_label_en,pred_label_en,gold_raw_full_label,exact_match,gold_num_pairs,pred_num_pairs,pred_entities,pred_aspect_categories
0,,"{RESTAURANT#GENERAL, positive}, {DRINKS#QUALIT...","{DRINKS#PRICES, neutral}, {DRINKS#QUALITY, pos...","{RESTAURANT#GENERAL, positive}, {DRINKS#QUALIT...",0,4,5,DRINKS | RESTAURANT,DRINKS#PRICES | DRINKS#QUALITY | DRINKS#STYLE&...
1,,"{LOCATION#GENERAL, neutral}, {RESTAURANT#MISCE...","{DRINKS#PRICES, positive}, {DRINKS#STYLE&OPTIO...","{LOCATION#GENERAL, neutral}, {RESTAURANT#MISCE...",0,7,8,DRINKS | FOOD | LOCATION | RESTAURANT,DRINKS#PRICES | DRINKS#STYLE&OPTIONS | FOOD#PR...
2,,"{FOOD#STYLE&OPTIONS, positive}, {AMBIENCE#GENE...","{AMBIENCE#GENERAL, positive}, {DRINKS#PRICES, ...","{FOOD#STYLE&OPTIONS, positive}, {AMBIENCE#GENE...",0,6,5,AMBIENCE | DRINKS,AMBIENCE#GENERAL | DRINKS#PRICES | DRINKS#QUAL...
3,,"{LOCATION#GENERAL, positive}, {RESTAURANT#PRIC...","{AMBIENCE#GENERAL, positive}, {DRINKS#PRICES, ...","{LOCATION#GENERAL, positive}, {RESTAURANT#PRIC...",0,8,10,AMBIENCE | DRINKS | FOOD | LOCATION | RESTAURA...,AMBIENCE#GENERAL | DRINKS#PRICES | DRINKS#QUAL...
4,,"{AMBIENCE#GENERAL, positive}, {RESTAURANT#PRIC...","{AMBIENCE#GENERAL, positive}, {FOOD#QUALITY, p...","{AMBIENCE#GENERAL, positive}, {RESTAURANT#PRIC...",0,5,4,AMBIENCE | FOOD | SERVICE,AMBIENCE#GENERAL | FOOD#QUALITY | FOOD#STYLE&O...
5,,"{AMBIENCE#GENERAL, positive}, {FOOD#QUALITY, p...","{AMBIENCE#GENERAL, positive}, {FOOD#QUALITY, p...","{AMBIENCE#GENERAL, positive}, {FOOD#QUALITY, p...",1,3,3,AMBIENCE | FOOD | RESTAURANT,AMBIENCE#GENERAL | FOOD#QUALITY | RESTAURANT#G...
6,,"{FOOD#STYLE&OPTIONS, positive}, {FOOD#QUALITY,...","{AMBIENCE#GENERAL, positive}, {FOOD#PRICES, ne...","{FOOD#STYLE&OPTIONS, positive}, {FOOD#QUALITY,...",0,5,6,AMBIENCE | FOOD | RESTAURANT | SERVICE,AMBIENCE#GENERAL | FOOD#PRICES | FOOD#QUALITY ...
7,,"{AMBIENCE#GENERAL, positive}, {FOOD#QUALITY, p...","{AMBIENCE#GENERAL, positive}, {FOOD#PRICES, po...","{AMBIENCE#GENERAL, positive}, {FOOD#QUALITY, p...",0,4,4,AMBIENCE | FOOD | SERVICE,AMBIENCE#GENERAL | FOOD#PRICES | FOOD#QUALITY ...
8,,"{RESTAURANT#GENERAL, positive}, {FOOD#QUALITY,...","{AMBIENCE#GENERAL, positive}, {FOOD#PRICES, ne...","{RESTAURANT#GENERAL, positive}, {FOOD#QUALITY,...",0,7,6,AMBIENCE | FOOD | LOCATION | RESTAURANT | SERVICE,AMBIENCE#GENERAL | FOOD#PRICES | FOOD#QUALITY ...
9,,"{LOCATION#GENERAL, negative}, {RESTAURANT#MISC...","{AMBIENCE#GENERAL, positive}, {DRINKS#PRICES, ...","{LOCATION#GENERAL, negative}, {RESTAURANT#MISC...",0,7,9,AMBIENCE | DRINKS | FOOD | LOCATION | RESTAURANT,AMBIENCE#GENERAL | DRINKS#PRICES | DRINKS#QUAL...


In [20]:
def debug_aspect_metrics_per_class(
    model, 
    loader, 
    split_name: str, 
    threshold: float = 0.5
) -> Dict[str, Dict[str, float]]:
    model.eval()
    all_true, all_probs = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"aspect-debug-{split_name}", leave=False):
            # Tuỳ chỉnh lại device sao cho khớp với code hiện tại của em
            input_ids = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()
            y_aspect = batch["y_aspect"].cpu().long()
            
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = out["aspect_probs"].cpu()
            
            all_true.append(y_aspect)
            all_probs.append(probs)

    y_true = torch.cat(all_true, dim=0)
    probs = torch.cat(all_probs, dim=0)
    y_pred = (probs >= threshold).long()

    print(f"\n===== Per-Aspect Metrics on {split_name} (Threshold: {threshold}) =====")
    print(f"{'Aspect Category':<30} | {'Precision':<9} | {'Recall':<9} | {'F1-Score':<9}")
    print("-" * 65)
    
    metrics_per_class = {}
    
    for i in range(probs.shape[1]):
        # Giả định id2aspect_category đã được định nghĩa ở global scope
        class_name = id2aspect_category[i] 
        
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        
        # Tính toán TP, FP, FN
        tp = (y_t * y_p).sum().float()
        fp = ((1 - y_t) * y_p).sum().float()
        fn = (y_t * (1 - y_p)).sum().float()
        
        # Tính metrics và xử lý lỗi chia cho 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else torch.tensor(0.0)
        recall = tp / (tp + fn) if (tp + fn) > 0 else torch.tensor(0.0)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else torch.tensor(0.0)
        
        p_val, r_val, f1_val = precision.item(), recall.item(), f1.item()
        
        metrics_per_class[class_name] = {
            "precision": p_val,
            "recall": r_val,
            "f1": f1_val
        }
        
        print(f"{class_name:<30} | {p_val:<9.4f} | {r_val:<9.4f} | {f1_val:<9.4f}")

    return metrics_per_class

with open(OUTPUT_DIR / "test_detailed_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_detailed, f, ensure_ascii=False, indent=2)

metrics_per_class = debug_aspect_metrics_per_class(best_model, test_loader, "test", 0.5)
metrics_per_class

aspect-debug-test:   0%|          | 0/16 [00:00<?, ?it/s]


===== Per-Aspect Metrics on test (Threshold: 0.5) =====
Aspect Category                | Precision | Recall    | F1-Score 
-----------------------------------------------------------------
AMBIENCE#GENERAL               | 0.8947    | 0.9333    | 0.9136   
DRINKS#PRICES                  | 0.7429    | 0.6842    | 0.7123   
DRINKS#QUALITY                 | 0.7284    | 0.8310    | 0.7763   
DRINKS#STYLE&OPTIONS           | 0.4533    | 0.7391    | 0.5620   
FOOD#PRICES                    | 0.8447    | 0.9698    | 0.9030   
FOOD#QUALITY                   | 0.9497    | 0.9912    | 0.9700   
FOOD#STYLE&OPTIONS             | 0.9045    | 0.9404    | 0.9221   
LOCATION#GENERAL               | 0.9017    | 0.8715    | 0.8864   
RESTAURANT#GENERAL             | 0.6172    | 0.8027    | 0.6979   
RESTAURANT#MISCELLANEOUS       | 0.9800    | 0.3769    | 0.5444   
RESTAURANT#PRICES              | 0.3898    | 0.3151    | 0.3485   
SERVICE#GENERAL                | 0.8621    | 0.8571    | 0.8596   


{'AMBIENCE#GENERAL': {'precision': 0.8947368264198303,
  'recall': 0.9333333373069763,
  'f1': 0.9136276245117188},
 'DRINKS#PRICES': {'precision': 0.7428571581840515,
  'recall': 0.6842105388641357,
  'f1': 0.7123287320137024},
 'DRINKS#QUALITY': {'precision': 0.7283950448036194,
  'recall': 0.8309859037399292,
  'f1': 0.7763157486915588},
 'DRINKS#STYLE&OPTIONS': {'precision': 0.4533333480358124,
  'recall': 0.739130437374115,
  'f1': 0.5619835257530212},
 'FOOD#PRICES': {'precision': 0.8447368144989014,
  'recall': 0.9697884917259216,
  'f1': 0.9029535055160522},
 'FOOD#QUALITY': {'precision': 0.9496855139732361,
  'recall': 0.9912472367286682,
  'f1': 0.9700214266777039},
 'FOOD#STYLE&OPTIONS': {'precision': 0.9045345783233643,
  'recall': 0.940446674823761,
  'f1': 0.9221411347389221},
 'LOCATION#GENERAL': {'precision': 0.9017341136932373,
  'recall': 0.8715083599090576,
  'f1': 0.8863636255264282},
 'RESTAURANT#GENERAL': {'precision': 0.617241382598877,
  'recall': 0.802690565586